# Direttore simple-service warehouse

Warehouse and Orders share one registry set and one root Unit of Work. The notebook imports the typed implementation from [`src/simple_service`](../../src/simple_service); it does not redefine application code. See [`docs/simple-service.md`](../../docs/simple-service.md) for the framework configuration reference.

In [1]:
from pathlib import Path
import sys

project_src = next(
    parent / 'src'
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / 'src/simple_service').exists()
)
sys.path.insert(0, str(project_src.resolve()))

from simple_service.application.errors import InsufficientStockError
from simple_service.application.inventory import (
    GetStockCommand, ReceiveStockCommand, RegisterProductCommand,
)
from simple_service.application.orders import PlaceOrderCommand
from simple_service.bootstrap.application import build_application
from simple_service.shared.lifecycle import RequestInput

## Bootstrap and validate

`build_application()` composes the holder factory, concrete UoW, populated registries, dependency container, saga journal, slot creator, and pool provider. The container maps only `StockReceiptClient` to its adapter—repositories are created by the concrete UoW.

In [2]:
example = build_application()
example.application.slot_provider_stats()

ExecutionSlotProviderStats(total_slots=1, free_slots=1, acquired_slots=0, max_slots=2)

## Direct and key-based commands

Register a product with a typed command. Receive stock through a stable registry key and payload. The request lifecycle turns optional input into the typed lifecycle context used by the handler.

In [3]:
request = RequestInput(actor_id='notebook', correlation_id='setup-100')
product = await example.application.handle(
    RegisterProductCommand('P-100', 'Keyboard'), input=request
)
balance = await example.application.handle_by_key(
    'warehouse.receive-stock.v1',
    {'product_id': 'P-100', 'quantity': 10},
)
product, balance, example.receipt_client.calls

(ProductSnapshot(product_id='P-100', name='Keyboard', quantity=0),
 StockBalance(product_id='P-100', quantity=10),
 [('P-100', 10)])

## Stored operation and one reused session

The loader resolves an operation ID through the active holder. Product reservation, order persistence, and the resulting event handler use the same session ID and commit once.

In [4]:
example.database.operations['place-100'] = (
    'orders.place-order.v1',
    {'order_id': 'O-100', 'product_id': 'P-100', 'quantity': 3},
)
access_start = len(example.database.access_log)
order = await example.application.handle_operation('place-100')
operation_accesses = example.database.access_log[access_start:]
order, operation_accesses, {session_id for _, session_id in operation_accesses}

(OrderSnapshot(order_id='O-100', product_id='P-100', quantity=3, status='placed'),
 [('operations.get', 3),
  ('products.reserve', 3),
  ('products.get', 3),
  ('orders.add', 3),
  ('movements.record', 3)],
 {3})

## Rollback on a domain error

An oversized order mutates no committed state. Direttore rolls back and closes the session.

In [5]:
try:
    await example.application.handle(PlaceOrderCommand('O-101', 'P-100', 8))
except InsufficientStockError as error:
    print(f'Expected failure: {error}')

stock = await example.application.handle(GetStockCommand('P-100'))
stock, 'O-101' in example.database.orders, example.database.transaction_log[-4:]

Expected failure: requested=8, available=7


(StockBalance(product_id='P-100', quantity=7),
 False,
 [('rollback', 4), ('close', 4), ('rollback', 5), ('close', 5)])

## Saga compensation

`ReceiveStockHandler` returns `SagaUseCaseHandlerResult`: its public result plus a typed `ReverseStockReceipt`. Its `compensate` method receives the application-specific saga context declared in `application/architecture.py`.

In [6]:
await example.application.handle(RegisterProductCommand('P-200', 'Mouse'))
await example.application.handle(
    ReceiveStockCommand('P-200', 2), saga_id='receipt-200'
)
before = await example.application.handle(GetStockCommand('P-200'))
await example.application.compensate_saga('receipt-200')
after = await example.application.handle(GetStockCommand('P-200'))
before, after

(StockBalance(product_id='P-200', quantity=2),
 StockBalance(product_id='P-200', quantity=0))

## Inspect observable effects

Restart the kernel and run all cells to reset the deterministic in-memory database.

In [7]:
example.database.products, example.database.orders, example.database.movements, example.application.slot_provider_stats()

({'P-100': {'product_id': 'P-100', 'name': 'Keyboard', 'quantity': 7},
  'P-200': {'product_id': 'P-200', 'name': 'Mouse', 'quantity': 0}},
 {'O-100': {'order_id': 'O-100',
   'product_id': 'P-100',
   'quantity': 3,
   'status': 'placed'}},
 [{'kind': 'stock_received',
   'product_id': 'P-100',
   'quantity': 10,
   'new_balance': 10},
  {'kind': 'order_placed',
   'order_id': 'O-100',
   'product_id': 'P-100',
   'quantity': 3},
  {'kind': 'stock_received',
   'product_id': 'P-200',
   'quantity': 2,
   'new_balance': 2}],
 ExecutionSlotProviderStats(total_slots=1, free_slots=1, acquired_slots=0, max_slots=2))